# 01 — The formula is the program

*The project's headline differentiator, end-to-end: write tensor expressions as LaTeX, parse them at compile time via the `_tex` user-defined literal, bind concrete tensors to the parsed AST through the `Evaluator`, and get back a `DynamicTensor` whose values are exactly what the formula says they should be.*

[ADR-0005](../docs/arc42/09-decisions/0005-adopt-tex-lyx-as-authoring-surface.md) calls TeX a first-class authoring surface. This notebook is what that means in practice.

> The library is **educational, not production**. For production needs, use Eigen / xtensor / libtorch / Kokkos / std::linalg. See [ADR-0001](../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md).

## Prerequisites

**[`00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals.** You need to know `Axis`, `Shape`, `Tensor<T, N>`, and Einstein-style broadcast ("`a_i + b_j → c_{ij}`"). No autograd, no GPU.

If you arrived here looking for compile-time-checked tensor algebra in C++ that reads like the math, this notebook is the demonstration. If you arrived here looking for autograd or GPU dispatch, those are notebooks [`05`](./05_autograd-from-scratch.ipynb) and [`06`](./06_webgpu-acceleration.ipynb).

## What this notebook covers

1. **Why TeX?** — the case for a LaTeX surface in a C++ tensor library.
2. **`_tex` UDL parse** — turning a string into an `Expression` AST at compile time.
3. **AST examination** — what the parser produces (`IndexedVar`, `BinOp`, `Sum`, `Equation`, `Group`).
4. **`Evaluator<T>`** — binding concrete tensors to AST leaves and running the expression.
5. **Outer product** — `c_{ij} = a_i b_j` end-to-end, reproducing the 00_intro headline.
6. **Einstein-sum** — `\sum_i {a_i b_i}` as a one-line inner-product expression.
7. **Round-trip property** — `to_latex(parse(s)) ≡ s` for the supported subset.
8. **Beyond the notebook** — the LyX export module (`lyx-export/`) so the *paper itself* is the program.

## Setup

Assumes the conda environment from `environment.yml` is active and the kernel is `xcpp20` (per [ADR-0014 §3](../docs/arc42/09-decisions/0014-external-substrate-strategy.md), xeus-cpp is the primary kernel; xeus-cling is kept as legacy smoke).

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <tensor/core/axis.hpp>
#include <tensor/core/shape.hpp>
#include <tensor/core/dynamic_shape.hpp>
#include <tensor/core/dynamic_tensor.hpp>
#include <tensor/core/format.hpp>
#include <tensor/tex/tex.hpp>

using tensor::core::Axis;
using tensor::core::DynamicShape;
using tensor::core::DynamicTensor;
using namespace tensor::tex::literals;

## 1 — Why TeX?

Tensor algebra is taught with a notation — Einstein notation, subscripted indices, `\sum`. Most C++ tensor libraries make you translate that notation into nested loops, indexed accessor functions, or template metaprogramming. The translation is lossy: the code does not look like the math; reviewers cannot diff a code change against a paper; bugs that would be obvious in notation hide in loop bounds.

The `_tex` UDL collapses the translation layer. The LaTeX subset the project supports — indexed variables, the four arithmetic operators with juxtaposition-as-multiplication, `\sum` over a labelled axis, equations — covers the operations a learner uses while building intuition for the algebra. Higher-order operators (tensor products with explicit indices, raising/lowering, gradients of tensor fields) are out of scope for this MVP; ADR-0005 frames them as future expansion if the LyX module sees use.

## 2 — `_tex` UDL parse

The simplest invocation: parse a string at compile time into an `Expression` AST. The result is structural — there is no evaluation yet, no values, no binding. Just the parse tree the project's tooling can consume.

In [ ]:
// `_tex` returns a tensor::tex::Expression; here we just round-trip it back
// to LaTeX via tensor::tex::to_latex(...) so we can sanity-check the parse.
auto expr = R"(c = a + b)"_tex;
std::cout << "input: c = a + b\n";
std::cout << "round-trip: " << tensor::tex::to_latex(expr) << '\n';

## 3 — AST examination

The `Expression` AST has five node kinds (see [`tensor::tex::Expression`](../include/tensor/tex/expression.hpp)):

- **`IndexedVar`** — a tensor reference with optional subscripts, e.g. `a`, `a_i`, `c_{ij}`.
- **`BinOp`** — one of `+`, `-`, `*`, `/`. Juxtaposition (e.g. `a_i b_j`) is parsed as multiplication.
- **`Sum`** — `\sum_i {body}`. The summation index `i` and the body are children.
- **`Equation`** — `lhs = rhs`. The LHS is an `IndexedVar` declaring the expected output shape; the RHS is the expression to evaluate.
- **`Group`** — explicit `{...}` braces, evaluation-transparent.

Round-trip is the easiest way to see what the parser produced; the printer always emits canonical form (explicit `{` braces, no implicit precedence).

In [ ]:
// Compare implicit multiplication to explicit:
auto implicit_mul = R"(a_i b_i)"_tex;     // juxtaposition
auto explicit_mul = R"(a_i * b_i)"_tex;  // explicit operator
std::cout << "implicit (`a_i b_i`) round-trips as: "
          << tensor::tex::to_latex(implicit_mul) << '\n';
std::cout << "explicit (`a_i * b_i`) round-trips as: "
          << tensor::tex::to_latex(explicit_mul) << '\n';

## 4 — `Evaluator<T>` — from formula to result

Parsing alone is structural. The `Evaluator<T>` ([`tensor/tex/evaluate.hpp`](../include/tensor/tex/evaluate.hpp)) closes the loop: bind named tensors to the AST's `IndexedVar` leaves, walk the expression, and produce a concrete `DynamicTensor<T>` as output. The walk delegates the actual algebra to `tensor::core`'s existing `DynamicTensor` operators — which already implement Einstein-style broadcast — so the parser and the evaluator share one notion of "what `a_i + b_j` means".

The simplest end-to-end example: bind two rank-1 tensors with disjoint labels, parse `c_{ij} = a_i + b_j`, evaluate.

In [ ]:
// Two rank-1 tensors with disjoint labels — the canonical broadcast setup.
DynamicTensor<double> a(DynamicShape{{Axis{"i", 3}}}, {1.0, 2.0, 3.0});
DynamicTensor<double> b(DynamicShape{{Axis{"j", 2}}}, {10.0, 20.0});

tensor::tex::Evaluator<double> ev;
ev.bind("a", a);
ev.bind("b", b);

auto sum_expr = R"(c_{ij} = a_i + b_j)"_tex;
auto c = ev.evaluate(sum_expr);
std::cout << "c_{ij} = a_i + b_j:\n" << c;

Output is rank-2 (axes `i` and `j`) with each entry $c_{ij} = a_i + b_j$. The shape is determined by the expression; the values are produced by the same machinery `00_intro.ipynb` §2 used directly on C++ tensors. The formula and the program agree by construction — there is no "check the loop indices" step a reviewer can get wrong.

## 5 — Outer product — reproducing the README's headline

Switch `+` to juxtaposition and the same setup becomes outer product: `c_{ij} = a_i b_j`. This is the example the [README](../README.md) leads with.

In [ ]:
auto outer_expr = R"(c_{ij} = a_i b_j)"_tex;
auto outer = ev.evaluate(outer_expr);
std::cout << "c_{ij} = a_i b_j:\n" << outer;

$c_{00} = 1 \cdot 10 = 10$, $c_{01} = 1 \cdot 20 = 20$, $c_{10} = 2 \cdot 10 = 20$, $c_{11} = 2 \cdot 20 = 40$, $c_{20} = 3 \cdot 10 = 30$, $c_{21} = 3 \cdot 20 = 60$. The output reads exactly like the math.

## 6 — Einstein-sum — a one-line inner product

When two tensors share an axis label, `\sum_i {a_i b_i}` is the Einstein-notation inner product. Bind two rank-1 tensors with the same label and the same extent, then parse + evaluate.

In [ ]:
DynamicTensor<double> c1(DynamicShape{{Axis{"i", 4}}}, {1.0, 2.0, 3.0, 4.0});
DynamicTensor<double> d1(DynamicShape{{Axis{"i", 4}}}, {2.0, 3.0, 5.0, 7.0});

tensor::tex::Evaluator<double> dot_ev;
dot_ev.bind("c", c1);
dot_ev.bind("d", d1);

auto inner_expr = R"(\sum_i {c_i d_i})"_tex;
auto inner = dot_ev.evaluate(inner_expr);
std::cout << "\\sum_i {c_i d_i} = \n" << inner;

Expected: $1 \cdot 2 + 2 \cdot 3 + 3 \cdot 5 + 4 \cdot 7 = 2 + 6 + 15 + 28 = 51$. The output is a rank-0 tensor holding the scalar `51`.

Behind the scenes the `Sum` AST node calls [`tensor::core::reduce_along_labels`](../include/tensor/core/reduce.hpp) on the result of evaluating the body, collapsing the `i` axis. The named-axis labels make this routing automatic — the `Sum` node does not need to be told which axis position to reduce over.

## 7 — Round-trip property

For every expression in the supported subset, `to_latex(parse(s))` returns a canonical form of `s` that re-parses to the same AST. This is the property the [`tensor::tex` detailed design doc](../docs/detailed-design/tensor-tex.md) commits to and the test suite verifies. It is what makes the `_tex` surface *invertible* — you can write LaTeX, the program runs, and you can recover (canonicalised) LaTeX from the AST for citation or display.

In [ ]:
// A few representative round-trips:
for (auto const& src : { R"(a_i)", R"(a_i + b_i)", R"(a_i b_j)",
                          R"(c_{ij} = a_i + b_j)",
                          R"(\sum_i {a_i b_i})" }) {
    auto ast = tensor::tex::parse(src);
    std::cout << "input : " << src << '\n';
    std::cout << "output: " << tensor::tex::to_latex(ast) << "\n\n";
}

## 8 — Beyond the notebook — LyX export

The `_tex` UDL handles strings. The [`lyx-export/`](../lyx-export/) module handles **documents**: a Python translator (`lyx_to_tex.py`) walks a LyX (`.lyx`) document, extracts every math-mode inset, and emits a `.cpp` file in which each inset becomes a `R"(...)"_tex` literal. A LyX module ([`tensor-math.module`](../lyx-export/tensor-math.module)) wires the translator into LyX's `File → Export → Tensor C++` menu so an author drafting a tensor-algebra paper can compile its formulas in one click.

This is the moment ADR-0005's slogan generalises from *the formula is the program* (inside a `.cpp` source) to *the paper is the program* (the whole LyX document compiles). Out of scope for this notebook to demonstrate end-to-end (LyX install required), but the [LyX plugin README](../lyx-export/LYX_PLUGIN.md) is a 10-minute setup.

## 9 — Where this fits

**Architecture:** `tensor::tex` is the project's DrivingAdapter ([ADR-0009](../docs/arc42/09-decisions/0009-adopt-ddd-ubiquitous-language-and-hexagonal-lite.md); see [arc42 §5](../docs/arc42/05-building-blocks/overview.md)). It consumes the Domain's `ExpressionSource` / `ExpressionSink` ports and produces / accepts expression graphs the `tensor::core` Domain executes. Switching the kernel backend (reference / Eigen / WebGPU) does not change the `_tex` surface at all — same formula, same parse, same evaluation, different kernels under the hood.

**Sibling notebooks:**
- [`00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals (this notebook's prerequisite).
- [`05_autograd-from-scratch.ipynb`](./05_autograd-from-scratch.ipynb) — autograd over named-axis tensors. Composes with `_tex` via the bound `DynamicVariable<T>` form.
- [`06_webgpu-acceleration.ipynb`](./06_webgpu-acceleration.ipynb) — WebGPU kernel backend; orthogonal to the `_tex` surface.
- [`07_mlp-on-toy.ipynb`](./07_mlp-on-toy.ipynb) — capstone training loop.
- [`08_swappable-backends.ipynb`](./08_swappable-backends.ipynb) — the Hexagonal-lite payoff for the reference + Eigen backend pair.

**Cross-references:**
- [`docs/detailed-design/tensor-tex.md`](../docs/detailed-design/tensor-tex.md) — implementation HOW (parser, AST, evaluator, LyX module).
- [ADR-0005](../docs/arc42/09-decisions/0005-adopt-tex-lyx-as-authoring-surface.md) — the decision that anchors this notebook.
- [arc42 §12 Glossary](../docs/arc42/12-glossary/overview.md) entries `_tex UDL`, `Expression`, `Evaluator`.
- The author's [2016 Qiita post](http://qiita.com/uyuutosa/items/12e87f4695bd151b1d74) — the spiritual ancestor; the named-axis intuition predates the LaTeX surface by a decade.